# Upload a Harbor dataset and run an evaluation

Publish three local tasks as a versioned taskset, run them through the Evaluator plugin, and inspect the saved results. No LLM or model API key is needed.

| Task | What to expect |
| --- | --- |
| `greet-universe` | The agent answers a greeting. |
| `sum-three` | The agent finishes but cannot do arithmetic: a low reward is expected. |
| `debug-agent-runtime-error` | An intentional runtime error demonstrates failed-trial reporting. |

## Before you start

- Use Python 3.12+ with the evaluator's Harbor extra and Jupyter installed in your repository environment.
- Follow the repository [setup guide](../../../../SETUP.md) to start Files, Entities, and Evaluator services and a host-subprocess evaluator worker. Docker must be available to the worker.
- The bundled wrapper is importable from the editable `nemo_evaluator` package.
- Open this notebook from its own directory or elsewhere in the repository. Run cells in order.

Uploads create persistent Tasks, a Taskset, and Files objects. Identical uploads can be repeated; submitting a job always starts a new evaluation.


In [8]:
import io
import json
import os
import re
import tarfile
import time
from pathlib import Path

from IPython.display import Markdown, display
from nemo_evaluator.sdk.harbor import register_harbor_dataset, upload_harbor_dataset
from nemo_platform_plugin.client.client import NemoClient
from nemo_platform_plugin.evaluator.client import EvaluatorClient
from nemo_platform_plugin.evaluator.types import SubmitAgentEvalJobRequest

# Edit these values, or set the corresponding environment variables.
BASE_URL = os.environ.get("NMP_BASE_URL", "http://localhost:8080")
WORKSPACE = os.environ.get("NMP_WORKSPACE", "default")
TASKSET_NAME = "hello-harbor-taskset"
AGENT_IMPORT_PATH = "nemo_evaluator.examples.harbor_test_agent:WrappedAgent"

cwd = Path.cwd().resolve()
EXAMPLE_DIR = next(
    (
        candidate
        for parent in (cwd, *cwd.parents)
        for candidate in (parent, parent / "plugins/nemo-evaluator/examples/harbor_taskset")
        if (candidate / "harbor_dataset").is_dir() and (candidate / "agent/harbor_wrapper.py").is_file()
    ),
    None,
)
if EXAMPLE_DIR is None:
    raise FileNotFoundError("Open this notebook from the example directory or repository checkout.")
DATASET = EXAMPLE_DIR / "harbor_dataset"

client = NemoClient(
    base_url=BASE_URL,
    workspace=WORKSPACE,
    auth=os.environ.get("NMP_API_KEY"),  # Only needed when your platform requires authentication.
)
evaluator_client = EvaluatorClient.from_client(client)
print(f"Platform: {BASE_URL} | Workspace: {WORKSPACE}")
print(f"Dataset: {DATASET}")

Platform: http://localhost:8080 | Workspace: default
Dataset: /Users/ngoncharenko/code/nemo-platform/plugins/nemo-evaluator/examples/harbor_taskset/harbor_dataset


## 1. Upload and register the dataset

Each task is stored as a self-contained `tar.gz` archive. The cell registers Tasks and a Taskset and returns immutable revision pins. Keep the taskset pin to evaluate the same definitions again.

This cell is rerunnable: if a Task or Taskset name already exists, registration retrieves and reuses that entity.


In [9]:
receipt = upload_harbor_dataset(DATASET, client=client, workspace=WORKSPACE, register=False)
receipt = register_harbor_dataset(
    receipt, client=client, workspace=WORKSPACE, taskset_name=TASKSET_NAME
)
assert receipt.taskset_ref is not None
TASKSET_REF = receipt.taskset_ref.root
print(f"Taskset pin: {TASKSET_REF}")
for member in receipt.members:
    assert member.task_ref is not None
    print(f"\n{member.native_name}")
    print(f"  Task:    {member.task_ref.root}")
    print(f"  Archive: {member.definition.source.fileset_ref}")

Taskset pin: default/hello-harbor-taskset#0a6da7a8b782a2fa0aa2498ad909e15bfca3c589e4fa5310d1589bba6cee1d7f

debug-agent-runtime-error
  Task:    default/debug-agent-runtime-error#bbe9c51ad490215c6f76705cdf06ef652d25846abf01c014bf18f6b939070c0b
  Archive: default/harbor-tasksets#harbor_dataset/debug-agent-runtime-error/4e0912b9ff75055eb723138dd436b170cef2122fd126df15dd75077fa82838bb/task_archive

greet-universe
  Task:    default/greet-universe#da7af148c05796552fdd86a26650ecba253b8b31802a3b353910e4282ef948f2
  Archive: default/harbor-tasksets#harbor_dataset/greet-universe/a713c036441147253b160e4e2e244b8efb6d58681ca1608bac09379c8a7665b1/task_archive

sum-three
  Task:    default/sum-three#eae1e6bd3f63ca3161df1eb6ab7582775d056db410fbd5c3b1e78fdaaf4de81c
  Archive: default/harbor-tasksets#harbor_dataset/sum-three/147ef57c5f605406aafa68eb59f7e59738d668cdc3e7fb437cfe02bfdfa9999c/task_archive


### What is stored in Files?

```text
FileSet: <workspace>/harbor-tasksets
└── harbor_dataset/
    ├── greet-universe/<archive-sha256>/task_archive
    ├── sum-three/<archive-sha256>/task_archive
    └── debug-agent-runtime-error/<archive-sha256>/task_archive
```

Each archive contains its named native task directory, including `task.toml`, instructions, environment, and tests. The archive hash identifies the compressed bytes; the Task and Taskset revision hashes identify database entity content. The taskset holds pinned Task references, not copies of their archives.


## 2. Submit an evaluation

The `tasks` field accepts the taskset pin returned above. The agent runs inside Docker task environments; its wrapper is provided by the evaluator package already loaded by the worker.

Run this cell once per desired evaluation. Polling and reading results below do not submit another job.


In [10]:
job = evaluator_client.submit_agent_eval_job(
    workspace=WORKSPACE,
    body=SubmitAgentEvalJobRequest(
        spec={
            "tasks": TASKSET_REF,
            "target": {
                "kind": "harbor",
                "agent_import_path": AGENT_IMPORT_PATH,
                "n_attempts": 1,
                "n_concurrent_trials": 1,
            },
        },
    ),
).data()
JOB_NAME = job.name
print(f"Submitted: {JOB_NAME}")

Submitted: nemo-evaluator.agent-zzd9szle


## 3. Wait for completion

Execution typically takes around 1–2 minutes; the first run may take longer while Docker images build. This cell prints status changes and waits up to 15 minutes. A notebook timeout does not cancel the job: rerun this cell to continue waiting for the same job.


In [ ]:
ansi_escape = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
def recent_job_logs():
    try:
        return evaluator_client.list_agent_eval_job_logs(
            workspace=WORKSPACE, name=JOB_NAME, query_params={"tail": 100}
        ).page().items
    except Exception:
        return []  # Status polling must still work if logs are temporarily unavailable.


deadline = time.monotonic() + 900
previous_status = None
while True:
    status = evaluator_client.get_agent_eval_job_status(
        workspace=WORKSPACE,
        name=JOB_NAME,
    ).data()
    if status.status.value != previous_status:
        print(f"{JOB_NAME}: {status.status.value}")
        previous_status = status.status.value
    if status.status.value in {"completed", "error", "cancelled"}:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError(f"{JOB_NAME} is still running. Rerun this cell to keep waiting.")
    time.sleep(2)

if status.status.value != "completed":
    logs = recent_job_logs()
    failed_tasks = [task for step in status.steps for task in step.tasks]
    causes = []
    for task in failed_tasks:
        error_stack = task.error_stack
        if error_stack is None:
            continue
        matches = re.findall(
            r"(?:ModuleNotFoundError|ValueError|RuntimeError|FileNotFoundError):[^|]+",
            error_stack,
        )
        causes.append(matches[-1].strip() if matches else error_stack[-500:].strip())
    error_message = (status.error_details or {}).get("message", "unknown error")
    display({"status": status.status.value, "error": error_message, "causes": causes})
    if logs:
        print("Recent job logs:")
        for log in logs[-8:]:
            message = ansi_escape.sub("", log.message).replace("\n", " ")
            print(f"  [{log.job_step}/{log.job_task}] {message[:500]}")
    cause = causes[-1] if causes else error_message
    raise RuntimeError(f"Job {JOB_NAME} ended with {status.status.value}: {cause}")

## 4. Read the saved results

A completed job means evaluation finished; it does **not** mean every task passed. The results bundle contains trial outcomes and scores. Read its JSONL members directly without extracting files onto your machine.


In [ ]:
payload = evaluator_client.download_agent_eval_job_result(
    workspace=WORKSPACE,
    job=JOB_NAME,
    name="agent-eval-results",
).read()

with tarfile.open(fileobj=io.BytesIO(payload), mode="r:*") as artifact:

    def read_records(filename):
        member = next(m for m in artifact.getmembers() if Path(m.name).name == filename)
        stream = artifact.extractfile(member)
        if stream is None:
            raise ValueError(f"Missing result file: {filename}")
        with stream:
            return [json.loads(line) for line in stream if line.strip()]

    trials = read_records("trials.jsonl")
    scores = read_records("scores.jsonl")

summary = ["| Task | Trial status | Score records |", "| --- | --- | --- |"]
for trial in trials:
    task_scores = [score for score in scores if score.get("task_id") == trial["task_id"]]
    summary.append(f"| {trial['task_id']} | {trial.get('status')} | {len(task_scores)} |")
display(Markdown("\n".join(summary)))
print(f"{len(trials)} trials, {len(scores)} scores | Job: {JOB_NAME}")

### Inspect scores and errors

Expand the structured records below to see reward values and the intentional runtime error. A completed arithmetic trial can still receive a zero reward.


In [ ]:
display({"scores": scores})
display(
    {
        "trial_errors": [
            {"task_id": trial["task_id"], "status": trial.get("status"), "error": trial.get("error")}
            for trial in trials
            if trial.get("error") or trial.get("status") != "completed"
        ]
    }
)

## Keep or clean up

Keep `TASKSET_REF` and `JOB_NAME` for inspection and repeat evaluations. Resources are retained by default. Tasksets do not own their members' archives, so do not delete shared Filesets to clean up this example.

The optional cell below deletes only this Taskset head; Tasks, archives, revisions, and job results remain.


In [ ]:
DELETE_TASKSET = False
if DELETE_TASKSET:
    from nemo_evaluator.sdk.taskset_resources import EvaluatorTasksetsResource

    EvaluatorTasksetsResource(evaluator_client).delete(TASKSET_NAME, workspace=WORKSPACE)
    print(f"Deleted Taskset head: {WORKSPACE}/{TASKSET_NAME}")